# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [1]:
!pip -q install pandas matplotlib

import json
import sqlite3
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# ------------------------------------------------------------
# Robust pack discovery for Colab / local runs
# ------------------------------------------------------------

def find_pack_root(search_root=Path("/content")):
    candidates = list(search_root.rglob("FlashEats_Classroom_Pack_V2"))
    for candidate in candidates:
        db = candidate / "database" / "flasheats.db"
        if db.exists():
            return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()
elif (Path.cwd().parent / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent   # solution notebooks live in a sub-folder of the pack

if BASE is None:
    BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload: FlashEats_Class6_Classroom_Pack.zip")
        uploaded = files.upload()

        zip_name = next(name for name in uploaded if name.endswith(".zip"))

        extract_dir = Path("/content/flasheats_class6")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root(Path("/content"))

    except Exception as e:
        print("Automatic Colab setup failed:", e)

# Local fallback: search current working directory too
if BASE is None:
    BASE = find_pack_root(Path.cwd())

print("Detected BASE:", BASE)

if BASE is None:
    raise FileNotFoundError(
        "Could not find FlashEats_Classroom_Pack_V2. "
        "Upload/extract the classroom pack ZIP, then rerun this cell."
    )

DB_PATH = BASE / "database" / "flasheats.db"

print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Detected BASE: C:\Users\harsh\Documents\SST\Forward Deployed Engineer\flasheats-classroom-pack
Database path: C:\Users\harsh\Documents\SST\Forward Deployed Engineer\flasheats-classroom-pack\database\flasheats.db
Database exists: True


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?


Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` is unique in `orders`; duplicates are identical | `orders.order_id.duplicated().sum() == 0`; compare column values across duplicate rows | **High**: duplicates are double-counted in both the numerator and the denominator, and conflicting copies make attributes such as traffic ambiguous |
| Delivered orders have completion time | Every `delivered` order has a non-null `actual_delivery_at` | Count delivered orders (after case-normalising status) with a null `actual_delivery_at` | **High**: those orders drop out of the denominator silently, which biases the rate |
| Promised ETA is valid | `promised_eta > created_at`, with a plausible promise window (e.g. 20–120 min) | `(promised_eta - created_at)` must be > 0; list the rows that fail | **High**: an impossible ETA creates fake, extreme delays |
| Event chronology is valid | `created_at ≤ pickup_at ≤ actual_delivery_at` | Pairwise timestamp comparisons | **Medium**: the order may still be late or on time, but the stage timings and the record itself are untrustworthy |
| “Late” has an agreed definition | One documented definition (threshold, population, exclusions) with a **named owner** | Read `client_metric_definitions.json`; ask who signs off | **Critical**: without it the number can be anything from 23% to 58% (see Challenge 2) |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

In [3]:
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print(orders["final_status"].value_counts(dropna=False))

# Solution cells below cover:
# - establish business grain
# - check delivered orders with missing completion time
# - test promised_eta >= created_at
# - test pickup_at <= actual_delivery_at
# - mark which assumptions need stakeholder clarification

Rows: 1603
Unique orders: 1600
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64


In [4]:
# Challenge 1 - run the validation contract checks
o = orders.copy()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    o[c] = pd.to_datetime(o[c], format="mixed", errors="coerce")
o["status_norm"] = o["final_status"].str.strip().str.lower()

# Grain: which columns disagree between duplicate copies?
dup_rows = o[o["order_id"].duplicated(keep=False)].sort_values("order_id")
conflicts = [c for c in o.columns if dup_rows.groupby("order_id")[c].nunique(dropna=False).max() > 1]
print("Duplicate order rows:", o["order_id"].duplicated().sum(), "| columns that conflict:", conflicts)
display(dup_rows[["order_id", "final_status", "traffic_bucket", "actual_delivery_at"]])

u = o.drop_duplicates("order_id").copy()          # order grain for the remaining checks
delivered = u[u["status_norm"] == "delivered"]

contract = pd.DataFrame([
    ("One row = one business order", "order_id unique", o["order_id"].duplicated().sum()),
    ("Delivered orders have completion time", "delivered -> actual_delivery_at not null", delivered["actual_delivery_at"].isna().sum()),
    ("Promised ETA is valid", "promised_eta > created_at", (u["promised_eta"] <= u["created_at"]).sum()),
    ("Event chronology is valid", "pickup_at >= created_at", (u["pickup_at"] < u["created_at"]).sum()),
    ("Event chronology is valid", "actual_delivery_at >= pickup_at", (u["actual_delivery_at"] < u["pickup_at"]).sum()),
    ("Status is consistent", "one spelling per status", (o["final_status"] != o["status_norm"]).sum()),
    ("'Late' has an agreed definition", "named KPI owner", metric_notes["note"]),
], columns=["assumption", "expectation", "violations"])
display(contract)

# Show the offending rows so owners can review them
display(u[u["promised_eta"] <= u["created_at"]][["order_id", "created_at", "promised_eta", "actual_delivery_at"]])
display(u[u["actual_delivery_at"] < u["pickup_at"]][["order_id", "created_at", "pickup_at", "actual_delivery_at"]])

Duplicate order rows: 3 | columns that conflict: ['traffic_bucket']


,order_id,final_status,traffic_bucket,actual_delivery_at
119,O00120,delivered,severe,2026-08-13 19:34:08.431508
1600,O00120,delivered,medium,2026-08-13 19:34:08.431508
722,O00723,delivered,medium,2026-08-05 22:04:30.928322
1601,O00723,delivered,high,2026-08-05 22:04:30.928322
1301,O01302,delivered,medium,2026-08-26 22:20:27.428561
1602,O01302,delivered,high,2026-08-26 22:20:27.428561


,assumption,expectation,violations
0,One row = one business order,order_id unique,3
1,Delivered orders have completion time,delivered -> actual_delivery_at not null,37
2,Promised ETA is valid,promised_eta > created_at,4
3,Event chronology is valid,pickup_at >= created_at,0
4,Event chronology is valid,actual_delivery_at >= pickup_at,5
5,Status is consistent,one spelling per status,5
6,'Late' has an agreed definition,named KPI owner,No canonical KPI owner is formally documented.


,order_id,created_at,promised_eta,actual_delivery_at
100,O00101,2026-08-24 20:59:00,2026-08-24 20:49:00,2026-08-24 22:14:30.685278
101,O00102,2026-08-11 16:41:00,2026-08-11 16:31:00,2026-08-11 17:57:17.661772
102,O00103,2026-08-22 20:11:00,2026-08-22 20:01:00,2026-08-22 21:10:17.342305
103,O00104,2026-08-09 22:19:00,2026-08-09 22:09:00,2026-08-09 23:36:20.740161


,order_id,created_at,pickup_at,actual_delivery_at
50,O00051,2026-08-01 15:09:00,2026-08-01 16:24:51.495665,2026-08-01 16:19:51.495665
51,O00052,2026-08-24 17:07:00,2026-08-24 18:16:16.946011,2026-08-24 18:11:16.946011
52,O00053,2026-08-13 17:06:00,2026-08-13 18:25:53.797613,2026-08-13 18:20:53.797613
53,O00054,2026-08-28 19:56:00,2026-08-28 21:25:10.582862,2026-08-28 21:20:10.582862
54,O00055,2026-08-16 12:53:00,2026-08-16 14:08:05.645770,2026-08-16 14:03:05.645770


### Challenge 1 — Answer

**What the contract checks found:**

| Assumption | Result | Needs stakeholder clarification? |
|---|---|---|
| One row = one order | **3 duplicate rows** (1,603 rows / 1,600 orders). The copies conflict only on `traffic_bucket` (severe vs medium, medium vs high). | Yes. The **source owner** must say which traffic value is true and fix the load (upsert vs append). |
| Delivered orders have completion time | **37 delivered orders** have no `actual_delivery_at` | Yes. **Data Team / Ops**: include, exclude, or backfill from driver events? |
| Promised ETA is valid | **4 orders** (O00101–O00104) have an ETA exactly **10 min before** the order was created | Yes. The **ETA/engineering owner** must say whether this is a bug and what the real promise was. |
| Chronology | **5 orders** (O00051–O00055) were delivered exactly **5 min before** pickup. No pickups happen before creation. | Yes, the owner must confirm. The pattern looks systematic, not random. |
| Status spelling | 5 rows say `Delivered` | No. This is only representation and is safe to normalise. |
| Agreed definition of “late” | **No canonical KPI owner is documented** and 4 stakeholders disagree | **Yes. This is the blocking issue.** |

**What could make 56% misleading:**
- The number itself is reproducible: 56.4% under "any delay > 0".
- But it hides three things:
  - a disputed definition
  - 105 orders outside the denominator (68 cancelled + 37 without a timestamp)
  - a few corrupted records
- None of these problems is fixed by cleaning. Each one needs an owner's decision.

# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|

Then answer:

> **Which one should leadership publish, and who must own that decision?**

In [5]:
analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (
    analysis["actual_delivery_at"] - analysis["promised_eta"]
).dt.total_seconds() / 60

# Solution: see the cells below

In [6]:
# Challenge 2 - same data, different definitions of "late"
analysis["status_norm"] = analysis["final_status"].str.strip().str.lower()
delivered_ok = analysis[(analysis["status_norm"] == "delivered") & analysis["actual_delivery_at"].notna()]
n_cancelled = (analysis["status_norm"] == "cancelled").sum()
n_late = (delivered_ok["delay_min"] > 0).sum()

# Historical dashboard style: raw rows (no dedupe), exact 'delivered', non-null delivery time
hist = orders[(orders["final_status"] == "delivered") & orders["actual_delivery_at"].notna()]
hist_delay = (pd.to_datetime(hist["actual_delivery_at"], format="mixed")
              - pd.to_datetime(hist["promised_eta"], format="mixed")).dt.total_seconds() / 60

definitions = pd.DataFrame([
    ("VP Operations: any delay > 0 min, delivered orders", n_late / len(delivered_ok), len(delivered_ok)),
    ("Support Lead: delay > 10 min, delivered orders", (delivered_ok["delay_min"] > 10).mean(), len(delivered_ok)),
    ("Data Team (historical): raw rows, exact 'delivered', non-null", (hist_delay > 0).mean(), len(hist)),
    ("Finance: exclude cancelled/refunded (refunds not in data)", n_late / len(delivered_ok), len(delivered_ok)),
    ("Customer promise: cancelled counted as failed", (n_late + n_cancelled) / (len(delivered_ok) + n_cancelled), len(delivered_ok) + n_cancelled),
], columns=["definition", "late_rate", "denominator"])
definitions["late_rate"] = (definitions["late_rate"] * 100).round(1)
display(definitions)

# Sensitivity: how fast does the rate fall as the threshold grows?
thresholds = pd.DataFrame({"threshold_min": [0, 5, 10, 15, 20]})
thresholds["late_rate_%"] = [round(100 * (delivered_ok["delay_min"] > t).mean(), 1) for t in thresholds["threshold_min"]]
display(thresholds)
print("Statuses present:", sorted(analysis["status_norm"].unique()), "-> no 'refunded' status exists")

,definition,late_rate,denominator
0,"VP Operations: any delay > 0 min, delivered orders",56.4,1495
1,"Support Lead: delay > 10 min, delivered orders",23.3,1495
2,"Data Team (historical): raw rows, exact 'delivered', non-null",56.3,1493
3,Finance: exclude cancelled/refunded (refunds not in data),56.4,1495
4,Customer promise: cancelled counted as failed,58.3,1563


,threshold_min,late_rate_%
0,0,56.4
1,5,37.5
2,10,23.3
3,15,13.9
4,20,8.2


Statuses present: ['cancelled', 'delivered'] -> no 'refunded' status exists


### Challenge 2 — Answer

| Definition | Late rate | Business meaning |
|---|---:|---|
| VP Ops: any delay > 0 min (delivered, deduped) | **56.4%** (843 / 1,495) | The promise was broken, even by one minute. This is the customer-promise view. |
| Support Lead: > 10 min | **23.3%** (349 / 1,495) | The delay was meaningful enough that customers are likely to notice or complain |
| Data Team historical (raw rows, exact `delivered`, non-null) | **56.3%** (n = 1,493) | Reproduces the dashboard. **This is almost certainly where "56%" came from.** It still includes duplicates and misses the capitalised `Delivered` rows. |
| Finance: exclude cancelled/refunded | 56.4%, **but incomplete** | Cancelled orders are already excluded. There is **no refunded status** in `orders`, so Finance's definition cannot be fully implemented. |
| Customer promise incl. cancelled as failed | **58.3%** | Every order where we did not deliver on time |

Moving the threshold from 0 → 5 → 10 → 15 min changes the rate from 56% → 37% → 23% → 14%. The **definition, not the data**, drives most of the variation.

**Which one should leadership publish, and who must own that decision?**
- **Recommendation:**
  - Publish the VP Ops definition as the headline KPI: *"% of delivered orders after the promised ETA" = 56.4%*. It matches the customer promise and the project question.
  - Always show a severity companion next to it: *">10 min late" = 23.3%*.
  - Always show cancellations and missing-timestamp orders as separate counts.
- **Owner:**
  - Leadership should not pick the number, and neither should the analyst. A **single named KPI owner**, logically the **VP Operations**, who is accountable for delivery performance, must sign off the definition: threshold, population, and treatment of cancelled, refunded and missing orders.
  - The Data Team is the steward that implements it.
  - Support and Finance are consulted.
  - The decision is written into `client_metric_definitions.json`.

# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [7]:
for col in ["final_status", "traffic_bucket"]:
    print("\n", col)
    print(orders[col].value_counts(dropna=False))

print("\nRestaurant status")
print(restaurant_status["status"].value_counts(dropna=False))

print("\nSupport category")
print(tickets["category"].value_counts(dropna=False))


 final_status
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

 traffic_bucket
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3
Name: count, dtype: int64

Restaurant status
status
preparing     171
handed_off    168
ready         157
ready           2
READY           1
Ready           1
handoff         1
unknown         1
Name: count, dtype: int64

Support category
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64


In [8]:
# Challenge 3 - raw values vs safe normalisation (trim + lowercase + space->underscore only)
fields = {
    "final_status": orders["final_status"],
    "traffic_bucket": orders["traffic_bucket"],
    "restaurant status": restaurant_status["status"],
    "ticket category": tickets["category"],
}
for name, s in fields.items():
    safe = s.str.strip().str.lower().str.replace(" ", "_")
    print(f"{name}: {s.nunique()} raw values -> {safe.nunique()} after safe normalisation")
    print("   raw:", s.map(repr).value_counts().to_dict())
    print("   safe:", safe.value_counts().to_dict(), "\n")

final_status: 3 raw values -> 2 after safe normalisation
   raw: {"'delivered'": 1530, "'cancelled'": 68, "'Delivered'": 5}
   safe: {'delivered': 1535, 'cancelled': 68} 

traffic_bucket: 5 raw values -> 4 after safe normalisation
   raw: {"'medium'": 635, "'high'": 448, "'low'": 381, "'severe'": 136, "'HIGH'": 3}
   safe: {'medium': 635, 'high': 451, 'low': 381, 'severe': 136} 

restaurant status: 8 raw values -> 5 after safe normalisation
   raw: {"'preparing'": 171, "'handed_off'": 168, "'ready'": 157, "'ready '": 2, "'READY'": 1, "'Ready'": 1, "'handoff'": 1, "'unknown'": 1}
   safe: {'preparing': 171, 'handed_off': 168, 'ready': 161, 'handoff': 1, 'unknown': 1} 

ticket category: 9 raw values -> 7 after safe normalisation
   raw: {"'late_delivery'": 38, "'eta_changed'": 37, "'restaurant_delay'": 37, "'ready_but_waiting'": 33, "'status_mismatch'": 29, "'driver_not_moving'": 25, "'Late Delivery'": 1, "'late_delivery '": 1, "'ETA issue'": 1}
   safe: {'late_delivery': 40, 'eta_change

In [9]:
# Semantic checks - things a trim/lowercase can't answer

# 1. Does each ticket label always carry the same customer message?
tk = tickets.assign(category_norm=tickets["category"].str.strip().str.lower().str.replace(" ", "_"))
display(tk.groupby(["category_norm", "customer_message"]).size().to_frame("tickets"))

# 2. What does the last restaurant status say for orders that were already delivered?
rs = restaurant_status.merge(orders.drop_duplicates("order_id")[["order_id", "final_status"]], on="order_id")
display(pd.crosstab(rs["status"].str.strip().str.lower(), rs["final_status"].str.lower()))

# 3. Records that need an owner, not a rule
display(restaurant_status[restaurant_status["status"].isin(["handoff", "unknown"])])

tickets
category_norm     customer_message                                                     
driver_not_moving The rider has not moved for 15 minutes.                            25
eta_changed       The ETA keeps changing and the food is still not here.             37
eta_issue         The rider has not moved for 15 minutes.                             1
late_delivery     My order is already past the promised time.                        39
                  The rider has not moved for 15 minutes.                             1
ready_but_waiting Restaurant says food is ready but no rider has picked it up.       33
restaurant_delay  Restaurant says they are still preparing my order.                 37
status_mismatch   The app says picking up but the restaurant says it is ready.       29

final_status,cancelled,delivered
status,,
handed_off,5,163
handoff,0,1
preparing,6,165
ready,4,157
unknown,0,1


,order_id,restaurant_id,status,last_updated_at
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


### Challenge 3 — Answer

| Field | Observed values | Representation (safe to normalise) | Semantics (owner must confirm) |
|---|---|---|---|
| `final_status` | `delivered` 1,530, `cancelled` 68, `Delivered` 5 | `Delivered` → `delivered` (casing only) | There is no `refunded` status, although Finance's definition needs one. Where are refunds recorded? |
| `traffic_bucket` | `medium`, `high`, `low`, `severe`, `HIGH` (3) | `HIGH` → `high` | Where is the boundary between `high` and `severe`, and how is it measured? The 3 duplicate orders carry *different* traffic values, so which one is true? |
| Restaurant `status` | `preparing` 171, `handed_off` 168, `ready` 157, `ready ` 2, `READY` 1, `Ready` 1, `handoff` 1, `unknown` 1 | All `ready` variants → `ready` (161) | **`handoff` vs `handed_off`:** "handoff in progress" and "handed off" may be different states, so do not merge by instinct. **`unknown`:** keep as unknown, do not impute. **`preparing`** is the *last* status for 165 orders that were delivered, so it may mean "never updated", not "still preparing". |
| Ticket `category` | 6 clean labels + `Late Delivery`, `late_delivery ` (trailing space), `ETA issue` | `Late Delivery` / `late_delivery ` → `late_delivery` | **`ETA issue` should not be mapped to `eta_changed`.** Its message is "The rider has not moved for 15 minutes", which matches `driver_not_moving`. The `Late Delivery` ticket carries the same "rider not moved" message, so the label and the text disagree. **The support owner** must say whether labels are agent-chosen or rule-based. |

**Rule applied:**
- Only casing and whitespace are normalised, and these changes are documented.
- Anything that changes the *meaning* of a record (merging states, re-labelling tickets, imputing `unknown`) is listed for the owner, not encoded silently.

# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

In [10]:
# Example:
# orders["restaurant_id"].dropna().isin(restaurants["restaurant_id"]).mean()

# Solution: see the cells below

In [11]:
# Challenge 4 - mapping coverage for each relationship
customers = pd.read_sql("SELECT * FROM customers", con)
order_grain = orders.drop_duplicates("order_id")

def coverage(child, key, parent, parent_key):
    # share of child rows whose key exists in the parent table
    nulls = child[key].isna().sum()
    matched = child[key].isin(parent[parent_key]).sum()
    return {"rows": len(child), "null keys": nulls, "unmatched keys": len(child) - nulls - matched,
            "coverage %": round(100 * matched / len(child), 2)}

relationships = {
    "orders.restaurant_id -> restaurants": coverage(order_grain, "restaurant_id", restaurants, "restaurant_id"),
    "orders.driver_id -> drivers": coverage(order_grain, "driver_id", drivers, "driver_id"),
    "orders.customer_id -> customers": coverage(order_grain, "customer_id", customers, "customer_id"),
    "tickets.order_id -> orders": coverage(tickets, "order_id", order_grain, "order_id"),
    "restaurant_status.order_id -> orders": coverage(restaurant_status, "order_id", order_grain, "order_id"),
}
display(pd.DataFrame(relationships).T)

,rows,null keys,unmatched keys,coverage %
orders.restaurant_id -> restaurants,1600.0,3.0,0.0,99.81
orders.driver_id -> drivers,1600.0,3.0,0.0,99.81
orders.customer_id -> customers,1600.0,0.0,0.0,100.00
tickets.order_id -> orders,202.0,3.0,0.0,98.51
restaurant_status.order_id -> orders,502.0,0.0,0.0,100.00


In [12]:
# Beyond "does the key exist": coverage in the other direction and consistency
print("Orders that have a restaurant_status record:",
      f"{order_grain['order_id'].isin(restaurant_status['order_id']).mean():.1%}")

chk = restaurant_status.merge(order_grain[["order_id", "restaurant_id"]], on="order_id", suffixes=("_status", "_order"))
print("restaurant_id disagreement between status and orders:", (chk["restaurant_id_status"] != chk["restaurant_id_order"]).sum())
print("Exact duplicate rows in restaurant_status:", restaurant_status.duplicated().sum())
print("Duplicate ticket rows:", tickets.duplicated().sum())

# Mapped, but the parent record itself is invalid (coordinates outside Bengaluru)
display(restaurants[~restaurants["lat"].between(12.5, 13.5) | ~restaurants["lon"].between(77.0, 78.5)])

# Driver mapping: orders keep the ORIGINAL driver; dispatch knows the current one
with open(BASE / "api" / "dispatch_data.json") as f:
    dispatch = pd.DataFrame(json.load(f))
d = order_grain.merge(dispatch[["order_id", "driver_id"]], on="order_id", suffixes=("_orders", "_dispatch"))
print("orders.driver_id != dispatch current driver:", (d["driver_id_orders"] != d["driver_id_dispatch"]).sum())

# Which orders are unmapped, and are they late?
unmapped = order_grain[order_grain["restaurant_id"].isna() | order_grain["driver_id"].isna()]
display(unmapped[["order_id", "restaurant_id", "driver_id", "final_status", "promised_eta", "actual_delivery_at"]])

Orders that have a restaurant_status record: 31.2%
restaurant_id disagreement between status and orders: 0
Exact duplicate rows in restaurant_status: 2
Duplicate ticket rows: 1


,restaurant_id,restaurant_name,cuisine,lat,lon,manual_status_updates
6,R007,Restaurant 007,North Indian,95.120000,77.694095,0
18,R019,Restaurant 019,Cafe,12.847954,190.330000,1


orders.driver_id != dispatch current driver: 98


,order_id,restaurant_id,driver_id,final_status,promised_eta,actual_delivery_at
400,O00401,NaN,D046,delivered,2026-08-24T19:23:36,2026-08-24T19:32:35.926083
401,O00402,NaN,D107,delivered,2026-08-21T18:52:49.033044,2026-08-21T18:47:25.022309
402,O00403,NaN,D099,delivered,2026-08-17T00:09:24,2026-08-17T00:26:52.711811
403,O00404,R010,NaN,delivered,2026-08-06T19:55:50.781031,2026-08-06T20:14:58.336586
404,O00405,R035,NaN,delivered,2026-08-20T22:10:59.969242,2026-08-20T21:58:43.608157
405,O00406,R048,NaN,delivered,2026-08-06T20:42:18,2026-08-06T20:29:35.614643


### Challenge 4 — Answer

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|
| `orders.restaurant_id` → restaurants | **99.81%** (3 null: O00401–O00403) | WARN | Harmless for the headline rate. Those 3 orders cannot be attributed to any restaurant. R007 and R019 map fine but have **impossible coordinates**, so any distance or geo analysis for them is wrong. |
| `orders.driver_id` → drivers | **99.81%** (3 null: O00404–O00406) | WARN | The key maps, but it is the *original* driver. **95 reassigned orders** point to the wrong driver compared with dispatch, so ~6% of driver attribution is wrong. |
| `orders.customer_id` → customers | 100% | PASS | None |
| `tickets.order_id` → orders | **98.51%** (3 null) + 1 duplicate ticket | WARN | 3 complaints are unlinkable and 1 is double-counted. This matters if we report "complaints per order" or tie tickets to late orders. |
| `restaurant_status.order_id` → orders | 100% (keys and restaurant_id agree) | PASS on keys / **FAIL on coverage** | Only **31%** of orders have any status record, and 2 rows are exact duplicates. Restaurant-level conclusions cover a minority of orders that may not be representative. |

**If 1% is unmapped, is that acceptable? It depends on the decision:**
- **Company-wide weekly late rate:** yes. 3 unmapped orders out of 1,495 cannot move 56.4% by more than 0.2 pts. Disclose it and move on.
- **Restaurant or driver accountability (rankings, penalties, payouts):** no.
  - A restaurant has only ~25 orders, so 1–3 missing or misattributed orders move its rate by 4–12 pts.
  - The driver key is wrong for 95 reassigned orders, not just the 3 nulls.
  - A rule that is fine for a dashboard can be unfair when a person's pay depends on it.
- **Restaurant status:** here the problem is not the 1% but that **69% of orders are unmapped** in the other direction.

# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [13]:
# Solution cells below cover:
# Merge restaurant_status with orders on order_id
# Parse timestamps
# Compare last_updated_at with order lifecycle timestamps

In [14]:
# Challenge 5 - how fresh is restaurant status relative to the order lifecycle?
import numpy as np

life = orders.drop_duplicates("order_id")[["order_id", "created_at", "pickup_at", "actual_delivery_at", "final_status"]].copy()
for c in ["created_at", "pickup_at", "actual_delivery_at"]:
    life[c] = pd.to_datetime(life[c], format="mixed", errors="coerce")

fs = restaurant_status.drop_duplicates().merge(life, on="order_id", how="left")
fs["last_updated_at"] = pd.to_datetime(fs["last_updated_at"])
fs["status_norm"] = fs["status"].str.strip().str.lower()

print("Orders with a status record:", fs["order_id"].nunique(), "of", life["order_id"].nunique())
print("Records per order:", fs.groupby("order_id").size().max(), "-> a single snapshot, no history")
print("Timestamps rounded to the minute:", f"{(fs['last_updated_at'].dt.second == 0).mean():.0%}")

# Where in the lifecycle did the last update happen?
fs["update_phase"] = np.select(
    [fs["last_updated_at"] < fs["created_at"],
     fs["last_updated_at"] <= fs["pickup_at"],
     fs["last_updated_at"] <= fs["actual_delivery_at"]],
    ["1 before order created", "2 created -> pickup", "3 pickup -> delivery"],
    "4 no delivery time")
display(pd.crosstab(fs["status_norm"], fs["update_phase"], margins=True))

# Minutes between the last status update and the pickup
fs["mins_before_pickup"] = (fs["pickup_at"] - fs["last_updated_at"]).dt.total_seconds() / 60
display(fs["mins_before_pickup"].describe().round(1).to_frame())
print("Last update > 15 min before pickup:", (fs["mins_before_pickup"] > 15).sum())
print("Last update after pickup:", (fs["mins_before_pickup"] < 0).sum())

Orders with a status record: 500 of 1600
Records per order: 1 -> a single snapshot, no history
Timestamps rounded to the minute: 100%


update_phase,1 before order created,2 created -> pickup,3 pickup -> delivery,4 no delivery time,All
status_norm,,,,,
handed_off,1,111,53,3,168
handoff,0,0,0,1,1
preparing,1,120,47,2,170
ready,3,97,57,3,160
unknown,0,1,0,0,1
All,5,329,157,9,500


,mins_before_pickup
count,500.0
mean,10.5
std,39.0
min,-25.4
25%,-3.2
50%,6.4
75%,16.8
max,385.8


Last update > 15 min before pickup: 142
Last update after pickup: 166


In [15]:
# Updates stamped hours BEFORE the order existed - stale record, or a timezone shift?
early = fs[fs["last_updated_at"] < fs["created_at"]].copy()
early["hours_before_created"] = ((early["created_at"] - early["last_updated_at"]).dt.total_seconds() / 3600).round(2)
early["mins_vs_created_if_utc_to_ist"] = (early["last_updated_at"] + pd.Timedelta(hours=5, minutes=30) - early["created_at"]).dt.total_seconds() / 60
display(early[["order_id", "status", "last_updated_at", "created_at", "hours_before_created", "mins_vs_created_if_utc_to_ist"]])

# Compare with the normal gap between order creation and status update
normal = fs[fs["last_updated_at"] >= fs["created_at"]]
print("Typical minutes from order created to status update:",
      ((normal["last_updated_at"] - normal["created_at"]).dt.total_seconds() / 60).median())

,order_id,status,last_updated_at,created_at,hours_before_created,mins_vs_created_if_utc_to_ist
5,O01016,handed_off,2026-08-17 08:01:00,2026-08-17 13:46:00,5.75,-15.0
6,O01395,ready,2026-08-21 09:45:00,2026-08-21 15:27:00,5.70,-12.0
7,O00925,preparing,2026-08-10 11:20:00,2026-08-10 17:10:00,5.83,-20.0
8,O00788,ready,2026-08-03 06:10:00,2026-08-03 11:58:00,5.80,-18.0
9,O01168,ready,2026-08-16 07:49:00,2026-08-16 13:43:00,5.90,-24.0


Typical minutes from order created to status update: 21.0


### Challenge 5 — Answer

**Findings:**
- **Coverage:** only **500 of 1,600 orders (31%)** have a status record.
- **One snapshot per order, no history:** we cannot see when a status *changed*, only the last value.
- **Precision:** 100% of timestamps are rounded to the minute, which suggests manual entry.
- **Staleness:**
  - 142 records (28%) were last updated **more than 15 min before pickup**.
  - 165 delivered orders still show **`preparing`** as their final status, so the status was never moved forward.
  - 166 records (33%) were last updated *after* pickup, so what the restaurant reports lags behind what actually happened.
- **Impossible timestamps:**
  - 5 orders were updated **~5.7–5.9 hours before the order existed**.
  - The obvious suspect is a UTC vs IST (+5:30) mix-up, but it does **not** explain them cleanly: after a +5:30 shift they are still 12–24 min *before* creation, while a normal update comes ~21 min *after*.
  - The cause is **UNKNOWN** (a stale record from an earlier order, a device clock, or another offset). The feed owner must investigate. We should not "fix" it by guessing an offset.

**The same data is judged against three different SLAs:**

| Use case | Freshness needed | Verdict | Why |
|---|---|---|---|
| **Weekly analytics** (trends, "which restaurants look slow") | Complete by the next day; minute precision is fine | **WARN** | Usable for direction only. 31% coverage (may not be random), no history, and 5 impossible timestamps to exclude. Every chart should state its coverage. |
| **Live customer ETA** | Every status transition within ~1–2 min of it happening, for every order | **FAIL** | Manual, sparse, a single snapshot, often stale (`preparing` after pickup). Showing this to customers would create exactly the `status_mismatch` and `ready_but_waiting` complaints we see in tickets. |
| **Restaurant accountability** (SLAs, penalties) | Complete, auditable history per order with trusted timestamps | **FAIL** | We cannot prove when the food was actually ready. Penalising restaurants from a manual, partial, possibly time-shifted field would be unfair and disputable. |

**What would make it fit:**
- system-generated, event-level status changes (`ACCEPTED`, `PREPARING`, `READY`, `HANDED_OFF`), each with its own timestamp in one timezone
- covering 100% of orders
- a freshness SLA and a monitored alert when updates stop

# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain | **WARN** | 1,603 rows / 1,600 orders; 3 duplicates conflict on `traffic_bucket` | Dedupe on `order_id` for the KPI. The source owner fixes the load (upsert) and confirms the true traffic values. |
| Timestamp chronology | **WARN** | 37 delivered orders without a delivery time; 4 ETAs 10 min *before* creation; 5 deliveries 5 min *before* pickup | Keep the 37 out of the denominator and disclose them. Flag the 9 bad-chronology orders: removing them only moves the rate from 56.4% to 56.3%. Engineering investigates the ETA and pickup bugs. The owner decides on backfilling the 37 from driver events. |
| KPI definition | **FAIL** | 4 stakeholder definitions give 23.3%–58.3%; no documented owner; Finance's "refunded" cannot even be computed | A named KPI owner (VP Ops) signs off threshold, population and exclusions in writing |
| Category semantics | **WARN** | Casing and whitespace are safe to fix; `handoff` vs `handed_off`, `unknown`, `ETA issue` and `preparing`-after-delivery are unresolved | Apply only the documented safe mappings. The restaurant-ops and support owners confirm the rest. |
| Cross-source mapping | **WARN** | 99.8% restaurant/driver keys map, 98.5% tickets; but 95 reassigned orders point to the original driver and only 31% of orders have restaurant status | OK for the company-wide rate. **Not** OK for driver or restaurant accountability until the current driver and full status coverage exist. |
| Freshness | **FAIL** (live / accountability) · **WARN** (weekly analytics) | Manual, minute-rounded single snapshots; 28% stale by >15 min; `preparing` after delivery; 5 updates stamped ~6 h before the order existed | Do not use restaurant status for live ETA or for penalties. Instrument system-generated status events. |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

What exactly must happen before the metric becomes publishable?

### Challenge 6 — Answer

**Should leadership publish "Late Delivery Rate = 56%" today? No, not in that form.**
- The number is *reproducible*: 56.4% of 1,495 delivered orders arrived after the promised ETA, and the old dashboard gives 56.3%.
- But it is **not yet defensible**:
  - Its definition has no owner.
  - A different, equally reasonable definition gives 23.3%.
  - It silently leaves out 105 orders.
- Publishing a bare "56%" invites exactly the argument this exercise surfaced.

**What must happen before it is publishable:**
1. **Owner sign-off (clears the FAIL):**
   - The VP Operations is named KPI owner.
   - They approve the definition in writing: late = `actual_delivery_at > promised_eta` (> 0 min), on unique delivered orders with a delivery timestamp. Cancelled orders are reported separately.
   - Finance says where refunds live.
2. **Fix or disclose the known defects:**
   - Dedupe the 3 orders.
   - List the 4 impossible ETAs and 5 broken chronologies. Excluding them gives 56.3%, so the headline holds.
   - Decide on the 37 missing delivery times (exclude, or backfill from driver events).
3. **Publish with context, not a bare number:**
   - *"56.4% of delivered orders missed the promised ETA (23.3% by more than 10 min); n = 1,495; 68 cancelled and 37 missing timestamps reported separately."*
4. **Automate the gate:**
   - The checks above run on every refresh.
   - The metric only publishes when no check is FAIL.

Until step 1 is done, the honest message to leadership is:
- *"Late delivery is somewhere between 23% and 58% depending on a definition nobody owns yet."*
- *"Under the VP Ops definition (any delay, delivered orders) it is 56%."*

In [16]:
validation_report = {
    "business_grain": "WARN - 3 duplicate order_ids (conflicting traffic_bucket); dedupe for KPI, owner fixes load",
    "timestamp_chronology": "WARN - 37 delivered w/o timestamp, 4 ETAs before creation, 5 delivered before pickup; exclude + disclose",
    "kpi_definition": "FAIL - 4 definitions (23.3% to 58.3%), no documented owner, 'refunded' not in data",
    "category_semantics": "WARN - casing/whitespace safe; handoff vs handed_off, unknown, 'ETA issue' need owners",
    "cross_source_mapping": "WARN - 99.8% keys map; 95 reassigned orders show original driver; status covers 31% of orders",
    "freshness": "FAIL for live ETA / accountability, WARN for weekly analytics - manual, stale, single snapshot",
    "publish_56_percent": "NO - reproducible (56.4%) but not publishable until the KPI owner signs off the definition"
}

for check, result in validation_report.items():
    print(f"{check:22} | {result}")

business_grain         | WARN - 3 duplicate order_ids (conflicting traffic_bucket); dedupe for KPI, owner fixes load
timestamp_chronology   | WARN - 37 delivered w/o timestamp, 4 ETAs before creation, 5 delivered before pickup; exclude + disclose
kpi_definition         | FAIL - 4 definitions (23.3% to 58.3%), no documented owner, 'refunded' not in data
category_semantics     | WARN - casing/whitespace safe; handoff vs handed_off, unknown, 'ETA issue' need owners
cross_source_mapping   | WARN - 99.8% keys map; 95 reassigned orders show original driver; status covers 31% of orders
freshness              | FAIL for live ETA / accountability, WARN for weekly analytics - manual, stale, single snapshot
publish_56_percent     | NO - reproducible (56.4%) but not publishable until the KPI owner signs off the definition


# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**